In [152]:
from pathlib import Path
import re
import json
from typing import List, Dict, Optional

In [153]:
def find_repo_root(start: Optional[Path] = None, repo_name: str = "masters_thesis_sdg") -> Path:
    """
    Walk upward from the current working directory until the repository root is found.
    """
    if start is None:
        start = Path.cwd().resolve()

    current = start
    while True:
        if current.name == repo_name:
            return current
        if current.parent == current:
            raise FileNotFoundError(
                f"Could not find repo root '{repo_name}' starting from {start}"
            )
        current = current.parent


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "lai2023"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Raw dir:", RAW_DIR)
print("Processed dir:", PROCESSED_DIR)

Repo root: C:\Users\annab\Documents\GitHub\masters_thesis_sdg
Raw dir: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\lai2023
Processed dir: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\processed


In [154]:
YT_TRANSCRIPTS_DIR = RAW_DIR / "Youtube" / "transcripts"
EGO4D_TRANSCRIPTS_DIR = RAW_DIR / "Ego4D" / "transcripts"

YT_OUTCOMES_DIR = RAW_DIR / "Youtube" / "vote_outcome_youtube_released"
EGO4D_OUTCOMES_DIR = RAW_DIR / "Ego4D" / "vote_outcome_ego4d"

OUTPUT_DIR = PROCESSED_DIR / "lai2023" / "onuw_transcripts_ready"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("YT_TRANSCRIPTS_DIR:", YT_TRANSCRIPTS_DIR)
print("EGO4D_TRANSCRIPTS_DIR:", EGO4D_TRANSCRIPTS_DIR)
print("YT_OUTCOMES_DIR:", YT_OUTCOMES_DIR)
print("EGO4D_OUTCOMES_DIR:", EGO4D_OUTCOMES_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

YT_TRANSCRIPTS_DIR: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\lai2023\Youtube\transcripts
EGO4D_TRANSCRIPTS_DIR: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\lai2023\Ego4D\transcripts
YT_OUTCOMES_DIR: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\lai2023\Youtube\vote_outcome_youtube_released
EGO4D_OUTCOMES_DIR: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\lai2023\Ego4D\vote_outcome_ego4d
OUTPUT_DIR: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\processed\lai2023\onuw_transcripts_ready


In [155]:
from pathlib import Path
import re
from collections import defaultdict

EGO4D_MERGED_DIR = PROCESSED_DIR / "lai2023" / "ego4d_merged_transcripts"
EGO4D_MERGED_DIR.mkdir(parents=True, exist_ok=True)

EGO4D_CHUNK_RE = re.compile(
    r"^(?P<session>.+?)_(?P<game>Game\d+)_(?P<chunk>\d+)\.txt$",
    re.IGNORECASE
)

EGO4D_SINGLE_RE = re.compile(
    r"^(?P<session>.+?)_(?P<game>Game\d+)\.txt$",
    re.IGNORECASE
)

In [156]:
def merge_ego4d_transcripts(input_dir: Path, output_dir: Path):
    grouped = defaultdict(list)
    passthrough_files = []
    failed = []

    for path in sorted(input_dir.rglob("*.txt")):
        name = path.name

        m_chunk = EGO4D_CHUNK_RE.match(name)
        if m_chunk:
            session = m_chunk.group("session")
            game = m_chunk.group("game")
            chunk = int(m_chunk.group("chunk"))
            grouped[(session, game)].append((chunk, path))
            continue

        m_single = EGO4D_SINGLE_RE.match(name)
        if m_single:
            passthrough_files.append(path)
            continue

        failed.append((path, "Filename did not match expected Ego4D patterns"))

    merged_count = 0
    copied_count = 0

    # Merge chunked files
    for (session, game), chunk_items in grouped.items():
        chunk_items = sorted(chunk_items, key=lambda x: x[0])
        merged_texts = []

        for chunk_id, chunk_path in chunk_items:
            text = chunk_path.read_text(encoding="utf-8").strip()
            if text:
                merged_texts.append(text)

        merged_text = "\n".join(merged_texts).strip() + "\n"
        out_path = output_dir / f"{session}_{game}.txt"
        out_path.write_text(merged_text, encoding="utf-8")
        merged_count += 1

    # Copy already-single files
    for path in passthrough_files:
        out_path = output_dir / path.name
        out_path.write_text(path.read_text(encoding="utf-8"), encoding="utf-8")
        copied_count += 1

    return {
        "merged_count": merged_count,
        "copied_count": copied_count,
        "failed": failed,
    }
merge_result = merge_ego4d_transcripts(
    input_dir=EGO4D_TRANSCRIPTS_DIR,
    output_dir=EGO4D_MERGED_DIR,
)

print("Merged games:", merge_result["merged_count"])
print("Copied single-file games:", merge_result["copied_count"])
print("Failed:", len(merge_result["failed"]))

if merge_result["failed"]:
    for fp, err in merge_result["failed"][:10]:
        print("-", fp, "->", err)

Merged games: 48
Copied single-file games: 0
Failed: 0


In [157]:
TIMESTAMPED_LINE_RE = re.compile(
    r"^\s*(?P<speaker>.+?)\s*\((?P<timestamp>\d{2}:\d{2})\)\s*:\s*(?P<utterance>.*)\s*$"
)

PLAIN_LINE_RE = re.compile(
    r"^\s*(?P<speaker>[^:]+?)\s*:\s*(?P<utterance>.*)\s*$"
)

TRANSCRIPT_FILENAME_RE = re.compile(
    r"^(?P<session>.+?)_(?P<game>Game\d+)\.txt$",
    re.IGNORECASE
)

ANNOUNCER_ALIASES = {
    "audio",
    "game audio",
    "automated",
    "siri voice",
    "siri",
    "timer",
    "twitch alert",
    "voiceover",
    "announcer",
}

# Optional: add corrections if you discover transcript misspellings
SPEAKER_CORRECTIONS = {
    # "Mitchel": "Mitchell",
}


def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def normalize_speaker(speaker: str) -> str:
    s = speaker.strip()

    if s in SPEAKER_CORRECTIONS:
        s = SPEAKER_CORRECTIONS[s]

    s_lower = s.lower()
    if s_lower in ANNOUNCER_ALIASES or s_lower.startswith("speaker "):
        return "Announcer"
    return s

In [158]:
def canonicalize_session_name(name: str) -> str:
    """
    Normalize transcript/outcome names so they can be matched robustly.
    Example:
    'Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#9'
    -> 'flashback one night ultimate werewolf retro 9'
    """
    name = Path(name).stem  # remove .json if present
    name = name.replace("#", " ")
    name = re.sub(r"[_\-]+", " ", name)
    name = re.sub(r"\s+", " ", name)
    return name.strip().lower()


def parse_transcript_filename(path: Path) -> Tuple[str, str]:
    """
    Example:
    some_session_Game3.txt -> ("some_session", "Game3")
    """
    m = TRANSCRIPT_FILENAME_RE.match(path.name)
    if not m:
        raise ValueError(f"Transcript filename does not match expected pattern: {path.name}")
    return m.group("session"), m.group("game")


def build_outcome_lookup(outcomes_dir: Path) -> Dict[str, Path]:
    """
    Build a map from canonicalized session name -> actual outcome json path.
    """
    lookup = {}
    for outcome_path in outcomes_dir.glob("*.json"):
        key = canonicalize_session_name(outcome_path.stem)
        if key in lookup:
            print(f"Warning: duplicate canonical outcome key '{key}'")
            print(" -", lookup[key])
            print(" -", outcome_path)
        lookup[key] = outcome_path
    return lookup


def load_outcome_json_for_session(session_name: str, outcome_lookup: Dict[str, Path]) -> Tuple[Path, Dict]:
    key = canonicalize_session_name(session_name)

    if key not in outcome_lookup:
        raise FileNotFoundError(
            f"Outcome file not found for session '{session_name}' "
            f"(canonical key: '{key}')"
        )

    outcome_path = outcome_lookup[key]

    with outcome_path.open("r", encoding="utf-8") as f:
        outcome_data = json.load(f)

    return outcome_path, outcome_data


def get_game_metadata(outcome_data: Dict, game_key: str) -> Dict:
    if game_key not in outcome_data:
        raise KeyError(f"{game_key} not found in outcome JSON.")
    return outcome_data[game_key]

In [159]:
def parse_transcript_line(line: str) -> Optional[Dict[str, str]]:
    line = line.strip()
    if not line:
        return None

    m = TIMESTAMPED_LINE_RE.match(line)
    if m:
        return {
            "speaker": normalize_speaker(m.group("speaker")),
            "utterance": normalize_whitespace(m.group("utterance")),
        }

    m = PLAIN_LINE_RE.match(line)
    if m:
        return {
            "speaker": normalize_speaker(m.group("speaker")),
            "utterance": normalize_whitespace(m.group("utterance")),
        }

    return None


def process_transcript_with_metadata(
    transcript_path: Path,
    source_name: str,
    player_names: List[str],
    game_metadata: Dict,
) -> Dict:
    """
    Keep only:
    - official players from outcome JSON
    - announcer-like lines -> normalized as 'Announcer'
    """
    allowed_players = set(player_names)

    raw_text = transcript_path.read_text(encoding="utf-8")
    raw_lines = raw_text.splitlines()

    kept_turns = []
    dropped_lines = []
    unparsed_lines = []

    for raw_line in raw_lines:
        parsed = parse_transcript_line(raw_line)

        if parsed is None:
            if raw_line.strip():
                unparsed_lines.append(raw_line.strip())
            continue

        speaker = parsed["speaker"]

        if speaker == "Announcer" or speaker in allowed_players:
            kept_turns.append(parsed)
        else:
            dropped_lines.append(raw_line.strip())

    for idx, turn in enumerate(kept_turns, start=1):
        turn["turn_id"] = idx

    session_name, game_key = parse_transcript_filename(transcript_path)

    formatted_lines = [
        f"Source: {source_name}",
        f"Session: {session_name}",
        f"Game: {game_key}",
        f"Players: {', '.join(player_names)}",
        "",
        "Transcript:"
    ]

    for turn in kept_turns:
        formatted_lines.append(
            f"[{turn['turn_id']}] {turn['speaker']}: {turn['utterance']}"
        )

    formatted_transcript = "\n".join(formatted_lines)

    return {
        "players": player_names,
        "turns": kept_turns,
        "formatted_transcript": formatted_transcript,
        "game_metadata": game_metadata,
        "unparsed_lines": unparsed_lines,
        "dropped_lines": dropped_lines,
    }

In [160]:
def to_repo_relative(path: Path, repo_root: Path) -> str:
    """
    Convert an absolute path to a POSIX-style path relative to the repo root.
    """
    return path.resolve().relative_to(repo_root.resolve()).as_posix()
def save_processed_game(
    transcript_path: Path,
    output_dir: Path,
    source_name: str,
    session_name: str,
    game_key: str,
    processed: Dict,
) -> Dict[str, Path]:
    source_output_dir = output_dir / source_name
    source_output_dir.mkdir(parents=True, exist_ok=True)

    base_name = f"{session_name}_{game_key}"
    txt_path = source_output_dir / f"{base_name}.txt"

    txt_path.write_text(processed["formatted_transcript"], encoding="utf-8")

    return {"txt_path": txt_path}

In [161]:
def process_source_dataset(
    source_name: str,
    transcripts_dir: Path,
    outcomes_dir: Path,
    output_dir: Path,
) -> Tuple[List[Dict], List[Tuple[Path, str]]]:
    transcript_files = sorted(transcripts_dir.rglob("*.txt"))
    outcome_lookup = build_outcome_lookup(outcomes_dir)

    print(f"[{source_name}] Found {len(transcript_files)} transcript files.")
    print(f"[{source_name}] Found {len(outcome_lookup)} outcome json files.")

    saved = []
    failed = []

    for transcript_path in transcript_files:
        try:
            session_name, game_key = parse_transcript_filename(transcript_path)
            outcome_path, outcome_data = load_outcome_json_for_session(session_name, outcome_lookup)
            game_metadata = get_game_metadata(outcome_data, game_key)

            player_names = game_metadata["playerNames"]

            processed = process_transcript_with_metadata(
                transcript_path=transcript_path,
                source_name=source_name,
                player_names=player_names,
                game_metadata=game_metadata,
            )

            out_paths = save_processed_game(
                transcript_path=transcript_path,
                output_dir=output_dir,
                source_name=source_name,
                session_name=session_name,
                game_key=game_key,
                processed=processed,
            )

            saved.append({
                "source": source_name,
                "session_name": session_name,
                "game_key": game_key,
                "source_transcript_file": to_repo_relative(transcript_path, REPO_ROOT),
                "source_outcome_file": to_repo_relative(outcome_path, REPO_ROOT),
                "processed_txt_path": to_repo_relative(out_paths["txt_path"], REPO_ROOT),
                "num_players": len(player_names),
                "num_turns": len(processed["turns"]),
                "num_unparsed_lines": len(processed["unparsed_lines"]),
                "num_dropped_lines": len(processed["dropped_lines"]),
                "player_names": player_names,
                "voting_outcome": game_metadata.get("votingOutcome"),
                "start_roles": game_metadata.get("startRoles"),
                "end_roles": game_metadata.get("endRoles"),
                "warning": game_metadata.get("warning"),
            })

        except Exception as e:
            failed.append((transcript_path, str(e)))

    print(f"[{source_name}] Processed successfully: {len(saved)}")
    print(f"[{source_name}] Failed: {len(failed)}")

    return saved, failed

In [162]:
all_saved = []
all_failed = []

yt_saved, yt_failed = process_source_dataset(
    source_name="Youtube",
    transcripts_dir=YT_TRANSCRIPTS_DIR,
    outcomes_dir=YT_OUTCOMES_DIR,
    output_dir=OUTPUT_DIR,
)
all_saved.extend(yt_saved)
all_failed.extend(yt_failed)

ego_saved, ego_failed = process_source_dataset(
    source_name="Ego4D",
    transcripts_dir=EGO4D_MERGED_DIR,
    outcomes_dir=EGO4D_OUTCOMES_DIR,
    output_dir=OUTPUT_DIR,
)
all_saved.extend(ego_saved)
all_failed.extend(ego_failed)

print("\nTOTAL PROCESSED:", len(all_saved))
print("TOTAL FAILED:", len(all_failed))

if all_failed:
    print("\nFirst failures:")
    for fp, err in all_failed[:10]:
        print(f"- {fp}: {err}")

[Youtube] Found 151 transcript files.
[Youtube] Found 57 outcome json files.
[Youtube] Processed successfully: 151
[Youtube] Failed: 0
[Ego4D] Found 48 transcript files.
[Ego4D] Found 11 outcome json files.
[Ego4D] Processed successfully: 48
[Ego4D] Failed: 0

TOTAL PROCESSED: 199
TOTAL FAILED: 0


In [163]:
index_path = OUTPUT_DIR / "index.json"
index_path.write_text(
    json.dumps(all_saved, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Saved index to:", index_path)

Saved index to: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\processed\lai2023\onuw_transcripts_ready\index.json


In [166]:
import json
from pathlib import Path

index_path = OUTPUT_DIR / "index.json"

with index_path.open("r", encoding="utf-8") as f:
    index_data = json.load(f)

non_na_warnings = []

for row in index_data:
    warning = row.get("warning", "N/A")
    if warning is None:
        continue
    if str(warning).strip().lower() not in {"n/a", "na", ""}:
        non_na_warnings.append({
            "source": row.get("source"),
            "session_name": row.get("session_name"),
            "game_key": row.get("game_key"),
            "warning": warning,
        })

print(f"Found {len(non_na_warnings)} non-N/A warnings.\n")

for item in non_na_warnings:
    print(
        f"[{item['source']}] {item['session_name']} - {item['game_key']}: {item['warning']}"
    )

Found 22 non-N/A warnings.

[Ego4D] 0a6ef9dc-a2dc-452b-a907-d6fa2ed4cae0 - Game1: Jack is the moderator and so did not vote
[Ego4D] 0a6ef9dc-a2dc-452b-a907-d6fa2ed4cae0 - Game2: Erin is the moderator. No one voted anyone
[Ego4D] 0c2659db-7bd4-4b37-9b08-4e247befe382 - Game5: Doppelganger becomes Insomniac and Ashley incorrecly votes for self
[Ego4D] 0c2659db-7bd4-4b37-9b08-4e247befe382 - Game7: Doppelganger becomes Robber
[Ego4D] 0c2659db-7bd4-4b37-9b08-4e247befe382 - Game8: Doppelganger becomes Robber
[Ego4D] 1a0d3e22-a6d6-4723-91a0-52fc2c87d5ce - Game10: Jacob incorrectly looks at center card
[Ego4D] 324bccd2-adff-4e9a-8ff4-03491e144ac3 - Game1: Hunter incorrectly checks own card at end of night phase
[Ego4D] 3ba069be-60fa-47fc-bd7b-f85bf649a5bd - Game1: Daniels new card was not shown, it was from the center. player 1, 2 and 4 voted middle
[Ego4D] 3ba069be-60fa-47fc-bd7b-f85bf649a5bd - Game2: player 1, 2 and 3 voted middle
[Ego4D] 3ba069be-60fa-47fc-bd7b-f85bf649a5bd - Game3: All play

In [167]:
import json
from collections import Counter

index_path = OUTPUT_DIR / "index.json"

with index_path.open("r", encoding="utf-8") as f:
    index_data = json.load(f)

start_role_counter = Counter()
end_role_counter = Counter()

for row in index_data:
    start_role_counter.update(row.get("start_roles", []))
    end_role_counter.update(row.get("end_roles", []))

all_roles = sorted(set(start_role_counter) | set(end_role_counter))

print("Unique roles in dataset:")
for role in all_roles:
    print(f"- {role}")

print("\nStart role counts:")
for role, count in start_role_counter.most_common():
    print(f"{role}: {count}")

print("\nEnd role counts:")
for role, count in end_role_counter.most_common():
    print(f"{role}: {count}")

Unique roles in dataset:
- Center Card
- Center card
- Doppelganger
- Doppleganger
- Drunk
- Hunter
- Insomniac
- Mason
- Minion
- Minions of Mordred
- Moderator
- NA
- Revealer
- Robber
- Seer
- Tanner
- Troublemaker
- Villager
- Werewolf

Start role counts:
Werewolf: 221
Robber: 118
Troublemaker: 115
Seer: 95
Tanner: 84
Villager: 70
Insomniac: 48
Drunk: 45
NA: 38
Minion: 27
Hunter: 20
Mason: 12
Moderator: 3
Doppelganger: 3
Doppleganger: 3
Minions of Mordred: 2

End role counts:
Werewolf: 229
Robber: 120
Troublemaker: 120
Seer: 99
Tanner: 84
Villager: 73
Insomniac: 49
NA: 38
Minion: 29
Hunter: 22
Drunk: 13
Mason: 11
Center card: 5
Moderator: 3
Doppelganger: 3
Doppleganger: 2
Minions of Mordred: 2
Revealer: 1
Center Card: 1


## Exclude invalid games

In [168]:
import json
from pathlib import Path
from collections import Counter

index_path = OUTPUT_DIR / "index.json"
clean_index_path = OUTPUT_DIR / "index_cleaned.json"
excluded_index_path = OUTPUT_DIR / "index_excluded.json"

with index_path.open("r", encoding="utf-8") as f:
    index_data = json.load(f)

ROLE_NORMALIZATION = {
    "Center card": "Center Card",
    "Doppleganger": "Doppelganger",
}

def normalize_role(role):
    if role is None:
        return role
    role = str(role).strip()
    return ROLE_NORMALIZATION.get(role, role)

def should_exclude_row(row):
    warning = str(row.get("warning", "")).strip().lower()
    if "different game" in warning:
        return True
    return False

clean_rows = []
excluded_rows = []
normalization_stats = Counter()

for row in index_data:
    new_row = dict(row)

    old_start_roles = row.get("start_roles", [])
    old_end_roles = row.get("end_roles", [])

    new_start_roles = []
    for r in old_start_roles:
        nr = normalize_role(r)
        if nr != r:
            normalization_stats[f"start_roles: {r} -> {nr}"] += 1
        new_start_roles.append(nr)

    new_end_roles = []
    for r in old_end_roles:
        nr = normalize_role(r)
        if nr != r:
            normalization_stats[f"end_roles: {r} -> {nr}"] += 1
        new_end_roles.append(nr)

    new_row["start_roles"] = new_start_roles
    new_row["end_roles"] = new_end_roles

    if should_exclude_row(new_row):
        excluded_rows.append(new_row)
    else:
        clean_rows.append(new_row)

with clean_index_path.open("w", encoding="utf-8") as f:
    json.dump(clean_rows, f, indent=2, ensure_ascii=False)

with excluded_index_path.open("w", encoding="utf-8") as f:
    json.dump(excluded_rows, f, indent=2, ensure_ascii=False)

print(f"Original rows: {len(index_data)}")
print(f"Cleaned rows kept: {len(clean_rows)}")
print(f"Excluded rows: {len(excluded_rows)}")

print("\nNormalizations applied:")
for k, v in normalization_stats.items():
    print(f"- {k}: {v}")

if excluded_rows:
    print("\nExcluded games:")
    for row in excluded_rows:
        print(f"- [{row['source']}] {row['session_name']} - {row['game_key']} | warning={row.get('warning')}")

Original rows: 199
Cleaned rows kept: 198
Excluded rows: 1

Normalizations applied:
- end_roles: Center card -> Center Card: 5
- start_roles: Doppleganger -> Doppelganger: 3
- end_roles: Doppleganger -> Doppelganger: 2

Excluded games:
- [Ego4D] 64076f60-f3fc-4ee6-93e1-3489d8ec7b12 - Game4 | warning=Different game


In [171]:
import json
from pathlib import Path
from collections import Counter

index_path = OUTPUT_DIR / "index.json"
clean_index_path = OUTPUT_DIR / "index_cleaned.json"
excluded_index_path = OUTPUT_DIR / "index_excluded.json"

with index_path.open("r", encoding="utf-8") as f:
    index_data = json.load(f)

ROLE_NORMALIZATION = {
    "Center card": "Center Card",
    "Doppleganger": "Doppelganger",
}

def normalize_role(role):
    if role is None:
        return role
    role = str(role).strip()
    return ROLE_NORMALIZATION.get(role, role)

def has_na_role(row):
    start_roles = row.get("start_roles", [])
    end_roles = row.get("end_roles", [])
    all_roles = [*start_roles, *end_roles]
    return any(str(r).strip().upper() == "NA" for r in all_roles)

def exclusion_reason(row):
    warning = str(row.get("warning", "")).strip().lower()
    if "different game" in warning:
        return "different_game"
    if has_na_role(row):
        return "na_role"
    return None

clean_rows = []
excluded_rows = []
normalization_stats = Counter()

for row in index_data:
    new_row = dict(row)

    old_start_roles = row.get("start_roles", [])
    old_end_roles = row.get("end_roles", [])

    new_start_roles = []
    for r in old_start_roles:
        nr = normalize_role(r)
        if nr != r:
            normalization_stats[f"start_roles: {r} -> {nr}"] += 1
        new_start_roles.append(nr)

    new_end_roles = []
    for r in old_end_roles:
        nr = normalize_role(r)
        if nr != r:
            normalization_stats[f"end_roles: {r} -> {nr}"] += 1
        new_end_roles.append(nr)

    new_row["start_roles"] = new_start_roles
    new_row["end_roles"] = new_end_roles

    reason = exclusion_reason(new_row)
    if reason is not None:
        new_row["exclusion_reason"] = reason
        excluded_rows.append(new_row)
    else:
        clean_rows.append(new_row)

with clean_index_path.open("w", encoding="utf-8") as f:
    json.dump(clean_rows, f, indent=2, ensure_ascii=False)

with excluded_index_path.open("w", encoding="utf-8") as f:
    json.dump(excluded_rows, f, indent=2, ensure_ascii=False)

print(f"Original rows: {len(index_data)}")
print(f"Cleaned rows kept: {len(clean_rows)}")
print(f"Excluded rows: {len(excluded_rows)}")

print("\nNormalizations applied:")
for k, v in normalization_stats.items():
    print(f"- {k}: {v}")

if excluded_rows:
    print("\nExcluded games:")
    for row in excluded_rows:
        print(
            f"- [{row['source']}] {row['session_name']} - {row['game_key']} "
            f"| reason={row.get('exclusion_reason')} | warning={row.get('warning')}"
        )

Original rows: 199
Cleaned rows kept: 191
Excluded rows: 8

Normalizations applied:
- end_roles: Center card -> Center Card: 5
- start_roles: Doppleganger -> Doppelganger: 3
- end_roles: Doppleganger -> Doppelganger: 2

Excluded games:
- [Ego4D] 0d495a56-676b-4bb0-bb7d-9abf88fb7beb - Game1 | reason=na_role | warning=NA
- [Ego4D] 0d495a56-676b-4bb0-bb7d-9abf88fb7beb - Game2 | reason=na_role | warning=NA
- [Ego4D] 0d495a56-676b-4bb0-bb7d-9abf88fb7beb - Game3 | reason=na_role | warning=NA
- [Ego4D] 0d495a56-676b-4bb0-bb7d-9abf88fb7beb - Game4 | reason=na_role | warning=NA
- [Ego4D] 1281ef80-2747-4bd1-8061-373409c879b5 - Game5 | reason=na_role | warning=NA
- [Ego4D] 1281ef80-2747-4bd1-8061-373409c879b5 - Game6 | reason=na_role | warning=NA
- [Ego4D] 1281ef80-2747-4bd1-8061-373409c879b5 - Game7 | reason=na_role | warning=NA
- [Ego4D] 64076f60-f3fc-4ee6-93e1-3489d8ec7b12 - Game4 | reason=different_game | warning=Different game


In [172]:
from collections import Counter

with clean_index_path.open("r", encoding="utf-8") as f:
    cleaned_index = json.load(f)

start_role_counter = Counter()
end_role_counter = Counter()

for row in cleaned_index:
    start_role_counter.update(row.get("start_roles", []))
    end_role_counter.update(row.get("end_roles", []))

all_roles = sorted(set(start_role_counter) | set(end_role_counter))

print("Unique roles after cleaning:")
for role in all_roles:
    print(f"- {role}")

Unique roles after cleaning:
- Center Card
- Doppelganger
- Drunk
- Hunter
- Insomniac
- Mason
- Minion
- Moderator
- Revealer
- Robber
- Seer
- Tanner
- Troublemaker
- Villager
- Werewolf


Player number distribution

In [175]:
import json
from collections import Counter

index_path = OUTPUT_DIR / "index_cleaned.json"  # or index.json if you prefer

with index_path.open("r", encoding="utf-8") as f:
    index_data = json.load(f)

num_players_counter = Counter()

for row in index_data:
    num_players = len(row.get("player_names", []))
    num_players_counter[num_players] += 1

print("Player count distribution across games:\n")
for n_players in sorted(num_players_counter):
    print(f"{n_players} players: {num_players_counter[n_players]} games")

Player count distribution across games:

3 players: 8 games
4 players: 100 games
5 players: 58 games
6 players: 25 games
